In [ ]:
import time
import utils_BiLSTMVersion_SavingImputedPathologiesVersion
import pandas as pd
import os
import torch
from torch import nn

device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Load the pickle files
scaled_train = pd.read_pickle("NIAvalueUpdatedVersion_scaled_whole_decedents_dataset.pkl")
# scaled_test = pd.read_pickle("NIAvalueUpdatedVersion_scaled_test.pkl")

In [ ]:
def run_lstm_hyperparameter_selection(feature_columns, target_columns, results_filename):
    start_time = time.time()

    train_ids = scaled_train.projid.unique()
    train_sequences = utils_BiLSTMVersion_SavingImputedPathologiesVersion.create_sequences(scaled_train, train_ids, feature_columns, target_columns)
    train_sequences = [(f.to(device), t.to(device)) for (f, t) in train_sequences]

    hyperparameter_grid = {
        'hidden_size': [32],
        'num_layers': [4],
        'learning_rate': [0.005],
        'batch_size': [32],
        'dropout_rate': [0]
    }

    results_df, oof_df = utils_BiLSTMVersion_SavingImputedPathologiesVersion.select_lstm_hyperparameters(
        train_sequences=train_sequences,
        feature_columns=feature_columns,
        target_columns=target_columns,
        hyperparameter_grid=hyperparameter_grid,
        seed=1217,
        n_splits=5,
        num_epochs=500,
        patience=10,
        lr_scheduler_patience=5,
        lr_factor=0.5,
        train_ids=train_ids,             # NEW
        df_all=scaled_train,             # NEW（To use the full decedent cohort, pass the corresponding complete data frame）
        device=device,
        bidirectional=True
    )

    os.makedirs("results", exist_ok=True)
    results_df.to_csv(results_filename, index=False)

    out_path = f"results/BiLSTM_nia_clean_imputed_{'_'.join(target_columns)}_cv5_allsteps.csv"
    oof_df.sort_values(["projid","fu_year"]).to_csv(out_path, index=False)

    print("Saved:", out_path)
    print(f"Results saved to {results_filename}")
    print(f"Time elapsed for target(s) {target_columns}: {time.time() - start_time:.2f} seconds")

In [ ]:
# Different target_columns
target_sets = [ 
    ['gpath', 'tangles', 'amyloid', 'niareagansc']
]

# Generate sequences for train and test sets
feature_columns = scaled_train.drop(columns=['projid', 'study', 'fu_year', 'cogdx', 
                                             'amyloid', 'gpath', 'tangles', 'niareagansc']).columns.tolist()

In [ ]:
os.makedirs('results', exist_ok=True)
for targets in target_sets:
    filename = f"results/Bilstm_nia_{'_'.join(targets)}.csv"  # Create unique filenames
    run_lstm_hyperparameter_selection(feature_columns, targets, filename)